# EMG Powerline Noise Removal — FIR vs IIR × Notch vs Comb

Interactive Jupyter notebook version of `filter_comparison.py`.

Use the **dropdown** in the last cell to choose a recording directory,
then click **▶ Run Analysis**.

### Pipeline
1. Load `emg_line_L.csv` from the chosen recording folder
2. **Bandpass 6–500 Hz** — 4th-order Butterworth, zero-phase (`sosfiltfilt`)
3. Apply **4 powerline-removal filters**, all zero-phase via `filtfilt`:

| # | Filter | Method |
|---|--------|--------|
| 1 | **FIR Notch** | `firwin` Hamming-window bandstop cascade |
| 2 | **IIR Notch** | `iirnotch` biquad cascade, Q = 30 |
| 3 | **FIR Comb** | Feedforward y[n] = x[n] − x[n−M] |
| 4 | **IIR Comb** | Feedback y[n] = x[n] − x[n−M] + r_m·y[n−M] |

4. Produce **6 figures** + a colour-coded **performance table**

> ⚠️ FIR Notch takes ~10 s at fs = 44 100 Hz (tap count capped at 4001).
> Set `SAVE_FIGS = True` in the Constants cell to also save figures as PNG files.


In [ ]:
%matplotlib inline

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as sig
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams.update({'figure.max_open_warning': 0})


In [ ]:
# Recording settings
RECORDINGS_DIR = Path("recordings")
DEFAULT_FOLDER = "emg_rec_20260625_092258"
CHANNEL        = "emg_line_L"

# Filter / signal parameters
F0            = 49.97328   # Hz -- actual mains frequency (measured, not exactly 50)
HARMONICS_MAX = 500.0      # Hz -- remove all harmonics up to this frequency
MAX_FIR_TAPS  = 4001       # cap to keep FIR Notch run-time finite at audio sample rates

# Set True to save each figure as a PNG file alongside the notebook
SAVE_FIGS = False

COLORS = {
    'raw':        '#AAAAAA',
    'bandpassed': '#1565C0',
    'FIR Notch':  '#E65100',
    'IIR Notch':  '#6A1B9A',
    'FIR Comb':   '#2E7D32',
    'IIR Comb':   '#C62828',
}

LABELS = {
    'FIR Notch': 'FIR Notch  (firwin, N<=4001, zero-phase)',
    'IIR Notch': 'IIR Notch  (iirnotch, Q=30, zero-phase)',
    'FIR Comb':  'FIR Comb   y[n]=x[n]-x[n-M]  (zero-phase)',
    'IIR Comb':  'IIR Comb   +r_m*y[n-M]  (zero-phase)',
}


In [ ]:
def load_recording(folder=DEFAULT_FOLDER, channel=CHANNEL,
                   recordings_dir=RECORDINGS_DIR):
    """Load one channel CSV. CSV has columns 'time' (s) and 'data' (V).
    fs is derived from the actual sample interval. Returns (t, x, fs)."""
    path = recordings_dir / folder / f"{channel}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Cannot find {path}")
    df = pd.read_csv(path)
    t  = df["time"].values.astype(float)
    x  = df["data"].values.astype(float)
    fs = 1.0 / (t[1] - t[0])
    print(f"Loaded  : {path}")
    print(f"Samples : {len(x):,}   fs = {fs:.4f} Hz   duration = {t[-1]:.3f} s")
    return t, x, fs


def bandpass_filter(x, fs, low_hz=6.0, high_hz=500.0, order=4):
    """Zero-phase 4th-order Butterworth bandpass 6-500 Hz via sosfiltfilt.
    Applied first to remove DC and electrode drift before notch/comb filters."""
    sos = sig.butter(order, [low_hz, high_hz], btype='band', fs=fs, output='sos')
    return sig.sosfiltfilt(sos, x)


def compute_fft(x, fs):
    """Single-sided FFT, returns (freqs, magnitude)."""
    N     = len(x)
    freqs = np.fft.rfftfreq(N, 1.0 / fs)
    mag   = np.abs(np.fft.rfft(x)) * (2.0 / N)
    mag[0] /= 2
    if N % 2 == 0:
        mag[-1] /= 2
    return freqs, mag


def single_stage_responses(fs, f0=F0, bw=5.0, Q=30.0, r_m=0.9, nfft=16384):
    """Compute single-stage magnitude (dB) for each filter type.
    Used for the frequency-response panels in plot_demo()."""
    M = int(round(fs / f0))

    n_taps = int(8 * fs / bw)
    if n_taps % 2 == 0:
        n_taps += 1
    if n_taps > MAX_FIR_TAPS:
        n_taps = MAX_FIR_TAPS if MAX_FIR_TAPS % 2 != 0 else MAX_FIR_TAPS - 1
    b_fn = sig.firwin(n_taps,
                      [max(f0 - bw / 2, 0.5), min(f0 + bw / 2, fs / 2 - 0.5)],
                      window='hamming', pass_zero=True, fs=fs)
    w_fn, H_fn = sig.freqz(b_fn, [1.0], worN=nfft, fs=fs)

    b_in, a_in = sig.iirnotch(f0, Q, fs=fs)
    w_in, H_in = sig.freqz(b_in, a_in, worN=nfft, fs=fs)

    b_fc = np.zeros(M + 1); b_fc[0] = 1.0; b_fc[M] = -1.0
    w_fc, H_fc = sig.freqz(b_fc, [1.0], worN=nfft, fs=fs)

    b_ic = np.zeros(M + 1); a_ic = np.zeros(M + 1)
    b_ic[0] = 1.0; b_ic[M] = -1.0
    a_ic[0] = 1.0; a_ic[M] = -r_m
    w_ic, H_ic = sig.freqz(b_ic, a_ic, worN=nfft, fs=fs)

    def dB(H):
        return 20 * np.log10(np.abs(H) + 1e-14)

    return {
        'FIR Notch': (w_fn, dB(H_fn)),
        'IIR Notch': (w_in, dB(H_in)),
        'FIR Comb':  (w_fc, dB(H_fc)),
        'IIR Comb':  (w_ic, dB(H_ic)),
    }


In [ ]:
def fir_notch_filter(x, fs, f0=F0, bw=5.0, max_hz=HARMONICS_MAX,
                     max_taps=MAX_FIR_TAPS):
    """
    FIR Notch -- cascade of Hamming-window bandstop filters, one per harmonic.
    Applied zero-phase via filtfilt.

    FIR = Finite Impulse Response = only ZEROS, no feedback, no poles.
    Transfer function: H(z) = b0 + b1*z^{-1} + ... + b_{N-1}*z^{-(N-1)}
    All N tap weights are symmetric -> linear phase.

    Trade-offs for EMG
    ------------------
    + Perfectly linear phase; unconditionally stable.
    - Needs N ~= 8*fs/bw taps per harmonic (Hamming window).
      At fs=44100 Hz, bw=5 Hz -> ~70 000 taps -- impractical!
    - Tap count cap (4001) gives ~88 Hz transition bands instead of 5 Hz.
    """
    n_taps = int(8 * fs / bw)
    if n_taps % 2 == 0:
        n_taps += 1
    capped = n_taps > max_taps
    if capped:
        n_taps = max_taps if max_taps % 2 != 0 else max_taps - 1
        actual_bw = 8 * fs / n_taps
        print(f"  [FIR Notch] tap count capped at {n_taps} "
              f"(ideal {int(8*fs/bw)+1}, transition band ~{actual_bw:.0f} Hz "
              f"instead of {bw} Hz -- notch very wide at fs={fs:.0f} Hz)")
    y = x.copy().astype(float)
    k = 1
    while k * f0 <= max_hz and k * f0 < fs / 2:
        fc = k * f0
        b  = sig.firwin(n_taps,
                        [max(fc - bw / 2, 0.5), min(fc + bw / 2, fs / 2 - 0.5)],
                        window='hamming', pass_zero=True, fs=fs)
        y  = sig.filtfilt(b, [1.0], y)
        k += 1
    return y


def iir_notch_filter(x, fs, f0=F0, Q=30.0, max_hz=HARMONICS_MAX):
    """
    IIR Notch -- cascade of 2nd-order biquad notch filters, one per harmonic.
    Applied zero-phase via filtfilt.

    IIR = Infinite Impulse Response = uses both ZEROS and POLES.
    Two zeros on the unit circle at +/-f_c -> infinite attenuation.
    Two poles just inside -> sharp, narrow notch; passband nearly untouched.

    Q = f0 / notch_bandwidth.  Q=30 -> ~1.7 Hz width at 50 Hz.

    Trade-offs for EMG
    ------------------
    + Sharp notch with only 2 poles + 2 zeros per harmonic.
    + Flat passband between harmonics -> minimum EMG distortion.
    + Same cost at any fs -- ideal for audio-rate EMG.
    - Not linear phase; filtfilt required for zero-phase (offline only).
    """
    y = x.copy().astype(float)
    k = 1
    while k * f0 <= max_hz and k * f0 < fs / 2:
        b, a = sig.iirnotch(k * f0, Q, fs=fs)
        y    = sig.filtfilt(b, a, y)
        k += 1
    return y


def fir_comb_filter(x, fs, f0=F0):
    """
    FIR Comb (feedforward): y[n] = x[n] - x[n-M], M = round(fs / f0).
    Applied zero-phase via filtfilt.

    Transfer function: H(z) = 1 - z^{-M}
    M zeros equally spaced on unit circle -> nulls at 0, f0, 2*f0, ..., (M-1)*f0.

    Trade-offs for EMG
    ------------------
    + Removes ALL harmonics in one pass -- ultra-efficient; linear phase.
    - Passband NOT flat: gain = 2|sin(pi*f*M/fs)|, squares to +12 dB with
      filtfilt between harmonics. DISTORTS the EMG significantly.
    """
    M    = int(round(fs / f0))
    b    = np.zeros(M + 1)
    b[0] = 1.0
    b[M] = -1.0
    return sig.filtfilt(b, [1.0], x.astype(float))


def iir_comb_filter(x, fs, f0=F0, r_m=0.9):
    """
    IIR Comb (feedback): y[n] = x[n] - x[n-M] + r_m * y[n-M]
    Transfer function: H(z) = (1 - z^{-M}) / (1 - r_m * z^{-M})

    r_m is the DIRECT feedback coefficient (fs-independent).
    BW_3dB ~= (1 - r_m) * f0 / pi  [Hz]
    r_m=0.90 -> BW ~= 1.6 Hz  |  r_m=0.98 -> BW ~= 0.3 Hz

    Trade-offs for EMG
    ------------------
    + Sharpest notches for the lowest cost; flat passband between harmonics.
    + r_m is fs-independent -- same value works at any sample rate.
    - Not linear phase; filtfilt achieves zero-phase for offline use.
    - Stability requires r_m < 1.
    """
    M    = int(round(fs / f0))
    b    = np.zeros(M + 1)
    a    = np.zeros(M + 1)
    b[0] = 1.0;  b[M] = -1.0
    a[0] = 1.0;  a[M] = -r_m
    bw   = (1 - r_m) * f0 / np.pi
    print(f"  [IIR Comb] M={M}, r_m={r_m}, approx notch BW={bw:.2f} Hz per harmonic")
    return sig.filtfilt(b, a, x.astype(float))


In [ ]:
def plot_demo(t, x_raw, x_bp, filtered, ffts, responses,
              fs=44100.0, f0=F0, folder=""):
    """Figure 1 -- main 5-row comparison figure."""
    fft_lim = min(530.0, fs / 2)
    f1_lo, f1_hi = f0 - 8, f0 + 8
    f2_lo, f2_hi = 2 * f0 - 8, 2 * f0 + 8

    fig = plt.figure(figsize=(18, 26), constrained_layout=True)
    fig.suptitle(
        f"EMG Powerline Noise Removal -- FIR vs IIR x Notch vs Comb\n"
        f"Recording: {folder}/{CHANNEL}.csv  |  "
        f"fs = {fs:.1f} Hz  |  f0 = {f0:.5f} Hz  |  "
        f"BP 6-500 Hz then 4 filters (all zero-phase via filtfilt)",
        fontsize=12, fontweight='bold',
    )
    gs = gridspec.GridSpec(5, 2, figure=fig,
                           height_ratios=[1.2, 1.1, 1.0, 1.1, 1.1])

    # (0,0): notch single-stage zoom
    ax = fig.add_subplot(gs[0, 0])
    ax.set_title(
        f"Freq response -- NOTCH (single stage, zoom +-15 Hz around f0)\n"
        f"FIR capped at {MAX_FIR_TAPS} taps -> transition band"
        f" ~{8*fs/MAX_FIR_TAPS:.0f} Hz wide (vs 5 Hz ideal)",
        fontsize=9,
    )
    for name in ('FIR Notch', 'IIR Notch'):
        w, H = responses[name]
        ax.plot(w, H, color=COLORS[name], lw=2.0, label=name)
    ax.axvline(f0, color='grey', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlim(f0 - 15, f0 + 15);  ax.set_ylim(-80, 3)
    ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (dB)")
    ax.legend(fontsize=9);  ax.grid(True, alpha=0.3)

    # (0,1): comb full spectrum
    ax = fig.add_subplot(gs[0, 1])
    ax.set_title(
        "Freq response -- COMB (single stage, 0 to 530 Hz)\n"
        "Both remove all harmonics at once; IIR has much narrower notches",
        fontsize=9,
    )
    for name in ('FIR Comb', 'IIR Comb'):
        w, H = responses[name]
        ax.plot(w, H, color=COLORS[name], lw=1.8, label=name)
    for k in range(1, int(fft_lim / f0) + 1):
        ax.axvline(k * f0, color='grey', ls=':', lw=0.5, alpha=0.35)
    ax.set_xlim(0, fft_lim);  ax.set_ylim(-80, 3)
    ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (dB)")
    ax.legend(fontsize=9);  ax.grid(True, alpha=0.3)

    # (1,:): all four around f0
    ax = fig.add_subplot(gs[1, :])
    ax.set_title(
        f"All four compared -- zoom {f0-15:.1f} to {f0+15:.1f} Hz  "
        "(IIR variants have flat passband; FIR Comb widest; FIR Notch very wide here)",
        fontsize=9,
    )
    for name in ('FIR Notch', 'IIR Notch', 'FIR Comb', 'IIR Comb'):
        w, H = responses[name]
        ax.plot(w, H, color=COLORS[name], lw=2.0, label=name)
    ax.axvline(f0, color='red', ls='--', lw=0.9, alpha=0.5,
               label=f'f0 = {f0:.5f} Hz')
    ax.set_xlim(f0 - 15, f0 + 15);  ax.set_ylim(-80, 3)
    ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (dB)")
    ax.legend(fontsize=9, ncol=3);  ax.grid(True, alpha=0.3)

    # (2,:): time domain excerpt
    ax  = fig.add_subplot(gs[2, :])
    tm  = (t >= 1.0) & (t < 1.5)
    ax.set_title(
        "Time domain -- 0.5 s excerpt  "
        "(grey = raw, dashed blue = bandpassed 6-500 Hz before harmonic removal)",
        fontsize=9,
    )
    ax.plot(t[tm], x_raw[tm], color=COLORS['raw'],        lw=0.5, alpha=0.55,
            label='Raw')
    ax.plot(t[tm], x_bp[tm],  color=COLORS['bandpassed'], lw=1.2, ls='--',
            label='Bandpassed 6-500 Hz')
    for name, y in filtered.items():
        ax.plot(t[tm], y[tm], color=COLORS[name], lw=0.9, label=LABELS[name])
    ax.set_xlabel("Time (s)");  ax.set_ylabel("Amplitude (V)")
    ax.legend(ncol=2, fontsize=7.5);  ax.grid(True, alpha=0.3)

    # (3,:): full FFT
    ax = fig.add_subplot(gs[3, :])
    ax.set_title(
        f"FFT magnitude -- 0 to {fft_lim:.0f} Hz (log scale)  "
        f"| red dotted = harmonics of f0={f0:.5f} Hz",
        fontsize=9,
    )
    fr_r, fft_r = ffts['raw']
    fr_b, fft_b = ffts['bandpassed']
    ax.semilogy(fr_r, fft_r, color=COLORS['raw'],        lw=0.5, alpha=0.4,
                label='Raw')
    ax.semilogy(fr_b, fft_b, color=COLORS['bandpassed'], lw=1.0, ls='--',
                label='Bandpassed')
    for name in ('FIR Notch', 'IIR Notch', 'FIR Comb', 'IIR Comb'):
        fr_f, fft_f = ffts[name]
        ax.semilogy(fr_f, fft_f, color=COLORS[name], lw=1.0, label=name)
    for k in range(1, int(HARMONICS_MAX / f0) + 2):
        ax.axvline(k * f0, color='red', ls=':', lw=0.5, alpha=0.22)
    ax.set_xlim(0, fft_lim)
    ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (log)")
    ax.legend(ncol=3, fontsize=8);  ax.grid(True, alpha=0.3, which='both')

    # (4,:): FFT zoom at f0 and 2*f0
    for col, (f_center, fl, fh) in enumerate([
        (f0,     f1_lo, f1_hi),
        (2 * f0, f2_lo, f2_hi),
    ]):
        ax = fig.add_subplot(gs[4, col])
        ax.set_title(f"FFT zoom: {f_center:.2f} Hz -- notch depth & width",
                     fontsize=9)
        ax.semilogy(fr_r, fft_r, color=COLORS['raw'],        lw=1.3, alpha=0.5,
                    label='Raw')
        ax.semilogy(fr_b, fft_b, color=COLORS['bandpassed'], lw=1.2, ls='--',
                    label='Bandpassed')
        for name in ('FIR Notch', 'IIR Notch', 'FIR Comb', 'IIR Comb'):
            fr_f, fft_f = ffts[name]
            zm = (fr_f >= fl) & (fr_f <= fh)
            ax.semilogy(fr_f[zm], fft_f[zm], color=COLORS[name], lw=1.3,
                        label=name)
        ax.axvline(f_center, color='red', ls=':', lw=0.9, alpha=0.5)
        ax.set_xlim(fl, fh)
        ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (log)")
        ax.legend(fontsize=7.5);  ax.grid(True, alpha=0.3, which='both')

    if SAVE_FIGS:
        fig.savefig("filter_comparison.png", dpi=150, bbox_inches='tight')
        print("Saved -> filter_comparison.png")
    plt.show()


In [ ]:
def plot_overlay(t, x_raw, x_bp, filtered, ffts,
                 fs=44100.0, f0=F0, folder=""):
    """Figure 2 -- full signal, all 6 cases overlaid: time (top) + FFT (bottom)."""
    fft_lim = min(530.0, fs / 2)
    step    = max(1, len(t) // 20000)

    cases = [
        ('Raw',                 x_raw,                'raw'),
        ('Bandpassed 6-500 Hz', x_bp,                 'bandpassed'),
        ('FIR Notch',           filtered['FIR Notch'], 'FIR Notch'),
        ('IIR Notch',           filtered['IIR Notch'], 'IIR Notch'),
        ('FIR Comb',            filtered['FIR Comb'],  'FIR Comb'),
        ('IIR Comb',            filtered['IIR Comb'],  'IIR Comb'),
    ]

    fig, (ax_t, ax_f) = plt.subplots(2, 1, figsize=(18, 10),
                                      constrained_layout=True)
    fig.suptitle(
        f"Full signal -- all cases overlaid  |  {folder}/{CHANNEL}\n"
        f"fs = {fs:.1f} Hz  |  f0 = {f0:.5f} Hz  |  duration = {t[-1]:.2f} s",
        fontsize=12, fontweight='bold',
    )

    ax_t.set_title("Full time domain -- all cases overlaid", fontsize=10)
    for label, y, key in cases:
        ax_t.plot(t[::step], y[::step], color=COLORS[key],
                  lw=0.35 if key == 'raw' else 0.65,
                  alpha=0.45 if key in ('raw', 'bandpassed') else 0.85,
                  label=label)
    ax_t.set_xlabel("Time (s)");  ax_t.set_ylabel("Amplitude (V)")
    ax_t.legend(ncol=3, fontsize=9);  ax_t.grid(True, alpha=0.3)

    ax_f.set_title(
        f"FFT -- all cases overlaid  (0-{fft_lim:.0f} Hz, log scale)"
        f"  |  red dotted = harmonics of f0",
        fontsize=10,
    )
    for label, _, key in cases:
        fr, mag = ffts[key]
        zm = fr <= fft_lim
        ax_f.semilogy(fr[zm], mag[zm], color=COLORS[key],
                      lw=0.5 if key == 'raw' else 1.0,
                      alpha=0.4 if key == 'raw' else 0.9,
                      label=label)
    for k in range(1, int(HARMONICS_MAX / f0) + 2):
        ax_f.axvline(k * f0, color='red', ls=':', lw=0.5, alpha=0.2)
    ax_f.set_xlim(0, fft_lim)
    ax_f.set_xlabel("Frequency (Hz)");  ax_f.set_ylabel("Magnitude (log)")
    ax_f.legend(ncol=3, fontsize=9);  ax_f.grid(True, alpha=0.3, which='both')

    if SAVE_FIGS:
        fig.savefig("filter_comparison_overlay.png", dpi=150, bbox_inches='tight')
        print("Saved -> filter_comparison_overlay.png")
    plt.show()


def plot_subplots_time(t, x_raw, x_bp, filtered,
                       fs=44100.0, f0=F0, folder=""):
    """Figure 3 -- full time domain, 3x2 separate subplots (one per case)."""
    step  = max(1, len(t) // 20000)
    cases = [
        ('Raw',                 x_raw,                'raw'),
        ('Bandpassed 6-500 Hz', x_bp,                 'bandpassed'),
        ('FIR Notch',           filtered['FIR Notch'], 'FIR Notch'),
        ('IIR Notch',           filtered['IIR Notch'], 'IIR Notch'),
        ('FIR Comb',            filtered['FIR Comb'],  'FIR Comb'),
        ('IIR Comb',            filtered['IIR Comb'],  'IIR Comb'),
    ]

    fig, axes = plt.subplots(3, 2, figsize=(18, 15), constrained_layout=True)
    fig.suptitle(
        f"Full time domain -- separate subplots  |  {folder}/{CHANNEL}\n"
        f"fs = {fs:.1f} Hz  |  duration = {t[-1]:.2f} s",
        fontsize=12, fontweight='bold',
    )
    for ax, (label, y, key) in zip(axes.flatten(), cases):
        rms = np.sqrt(np.mean(y ** 2))
        ax.plot(t[::step], y[::step], color=COLORS[key], lw=0.45)
        ax.set_title(label, fontsize=10, fontweight='bold', color=COLORS[key])
        ax.set_xlabel("Time (s)");  ax.set_ylabel("Amplitude (V)")
        ax.grid(True, alpha=0.3)
        ax.text(0.99, 0.97, f"RMS = {rms:.5f} V",
                transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color='dimgrey')

    if SAVE_FIGS:
        fig.savefig("filter_comparison_time_subplots.png", dpi=150,
                    bbox_inches='tight')
        print("Saved -> filter_comparison_time_subplots.png")
    plt.show()


def plot_subplots_fft(ffts, fs=44100.0, f0=F0, folder=""):
    """Figure 4 -- FFT log scale, 3x2 separate subplots (one per case)."""
    fft_lim    = min(530.0, fs / 2)
    case_order = [
        ('Raw',                 'raw'),
        ('Bandpassed 6-500 Hz', 'bandpassed'),
        ('FIR Notch',           'FIR Notch'),
        ('IIR Notch',           'IIR Notch'),
        ('FIR Comb',            'FIR Comb'),
        ('IIR Comb',            'IIR Comb'),
    ]

    fig, axes = plt.subplots(3, 2, figsize=(18, 15), constrained_layout=True)
    fig.suptitle(
        f"FFT -- separate subplots  (0-{fft_lim:.0f} Hz, log scale)"
        f"  |  {folder}/{CHANNEL}\n"
        f"Red dotted lines = harmonics of f0 = {f0:.5f} Hz",
        fontsize=12, fontweight='bold',
    )
    for ax, (label, key) in zip(axes.flatten(), case_order):
        fr, mag = ffts[key]
        zm = fr <= fft_lim
        ax.semilogy(fr[zm], mag[zm], color=COLORS[key], lw=0.8)
        for k in range(1, int(HARMONICS_MAX / f0) + 2):
            ax.axvline(k * f0, color='red', ls=':', lw=0.6, alpha=0.3)
        ax.set_title(label, fontsize=10, fontweight='bold', color=COLORS[key])
        ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (log)")
        ax.set_xlim(0, fft_lim);  ax.grid(True, alpha=0.3, which='both')
        mask_f0 = (fr >= f0 - 2) & (fr <= f0 + 2)
        if mask_f0.any():
            peak_v = float(np.max(mag[mask_f0]))
            ax.annotate(
                f"f0 peak\n{20*np.log10(peak_v+1e-20):.1f} dBV",
                xy=(f0, peak_v), xytext=(f0 + 15, peak_v * 3),
                fontsize=7, color='red', alpha=0.7,
                arrowprops=dict(arrowstyle='->', color='red', lw=0.7),
            )

    if SAVE_FIGS:
        fig.savefig("filter_comparison_fft_subplots.png", dpi=150,
                    bbox_inches='tight')
        print("Saved -> filter_comparison_fft_subplots.png")
    plt.show()


In [ ]:
def plot_subplots_fft_linear(ffts, fs=44100.0, f0=F0, folder=""):
    """Figure 5 -- FFT linear scale, 0-800 Hz, 3x2 subplots.
    Linear scale makes absolute spike heights directly comparable."""
    fft_lim    = min(800.0, fs / 2)
    case_order = [
        ('Raw',                 'raw'),
        ('Bandpassed 6-500 Hz', 'bandpassed'),
        ('FIR Notch',           'FIR Notch'),
        ('IIR Notch',           'IIR Notch'),
        ('FIR Comb',            'FIR Comb'),
        ('IIR Comb',            'IIR Comb'),
    ]

    fig, axes = plt.subplots(3, 2, figsize=(18, 15), constrained_layout=True)
    fig.suptitle(
        f"FFT -- linear amplitude scale  (0-{fft_lim:.0f} Hz)"
        f"  |  {folder}/{CHANNEL}\n"
        f"Red dotted lines = harmonics of f0 = {f0:.5f} Hz",
        fontsize=12, fontweight='bold',
    )
    for ax, (label, key) in zip(axes.flatten(), case_order):
        fr, mag = ffts[key]
        zm = fr <= fft_lim
        ax.plot(fr[zm], mag[zm], color=COLORS[key], lw=0.8)
        for k in range(1, int(fft_lim / f0) + 2):
            ax.axvline(k * f0, color='red', ls=':', lw=0.6, alpha=0.3)
        ax.set_title(label, fontsize=10, fontweight='bold', color=COLORS[key])
        ax.set_xlabel("Frequency (Hz)");  ax.set_ylabel("Magnitude (V/bin)")
        ax.set_xlim(0, fft_lim);  ax.grid(True, alpha=0.3)
        mask_f0 = (fr >= f0 - 2) & (fr <= f0 + 2)
        if mask_f0.any():
            peak_v = float(np.max(mag[mask_f0]))
            ax.annotate(
                f"f0 = {peak_v:.4f}",
                xy=(f0, peak_v),
                xytext=(f0 + 30, peak_v * 0.85 + ax.get_ylim()[1] * 0.05),
                fontsize=7, color='red', alpha=0.8,
                arrowprops=dict(arrowstyle='->', color='red', lw=0.7),
            )

    if SAVE_FIGS:
        fig.savefig("filter_comparison_fft_linear.png", dpi=150,
                    bbox_inches='tight')
        print("Saved -> filter_comparison_fft_linear.png")
    plt.show()


def plot_performance_table(x_bp, filtered, timings, ffts,
                            fs=44100.0, f0=F0, folder=""):
    """
    Figure 6 -- colour-coded performance summary table.
      Rows : Bandpassed (ref) + 4 filters
      Cols : RMS (V), Time (s), attenuation (dB) at every harmonic up to 500 Hz

    Attenuation cell colours:
      Dark green  >20 dB  |  Light green 10-20 dB  |  Yellow 3-10 dB
      Light red   0-3 dB  |  Red <0 dB (amplification)
    RMS cell: green within +-10% of bandpassed; yellow +-50%; red otherwise.
    """
    def _peak(key, f_target, f_bw=1.5):
        fr, mag = ffts[key]
        mask = (fr >= f_target - f_bw) & (fr <= f_target + f_bw)
        return float(np.max(mag[mask])) if mask.any() else 1e-20

    harmonics = []
    k = 1
    while k * f0 <= HARMONICS_MAX:
        harmonics.append(k * f0)
        k += 1

    ref_peaks = [_peak('bandpassed', h) for h in harmonics]
    rms_bp    = float(np.sqrt(np.mean(x_bp ** 2)))

    rows_text  = []
    rows_color = []

    def _atten_color(db):
        if db > 20:  return '#A5D6A7'
        if db > 10:  return '#C8E6C9'
        if db > 3:   return '#FFF9C4'
        if db >= 0:  return '#FFCDD2'
        return '#EF9A9A'

    def _rms_color(rms):
        ratio = abs(rms / rms_bp - 1)
        if ratio <= 0.10: return '#C8E6C9'
        if ratio <= 0.50: return '#FFF9C4'
        return '#FFCDD2'

    ref_text  = ['Bandpassed (ref)', f'{rms_bp:.5f}', '---']
    ref_color = ['#E3F2FD', '#C8E6C9', '#E3F2FD']
    for _ in harmonics:
        ref_text.append('(ref)')
        ref_color.append('#E3F2FD')
    rows_text.append(ref_text)
    rows_color.append(ref_color)

    for name, y in filtered.items():
        rms  = float(np.sqrt(np.mean(y ** 2)))
        t_s  = timings.get(name, float('nan'))
        row_t = [name, f'{rms:.5f}', f'{t_s:.2f} s']
        row_c = ['#FAFAFA', _rms_color(rms), '#FAFAFA']
        for h, rp in zip(harmonics, ref_peaks):
            p     = _peak(name, h)
            atten = 20 * np.log10(rp / (p + 1e-20))
            row_t.append(f'{atten:+.1f} dB')
            row_c.append(_atten_color(atten))
        rows_text.append(row_t)
        rows_color.append(row_c)

    harm_cols  = [f'{k+1}xf0\n{h:.1f} Hz' for k, h in enumerate(harmonics)]
    col_labels = ['Filter', 'RMS (V)', 'Time'] + harm_cols
    n_rows = len(rows_text)
    n_cols = len(col_labels)

    fig_w = max(16, 2 + n_cols * 1.3)
    fig_h = max(4,  1 + n_rows * 0.7)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    fig.suptitle(
        f"Filter performance summary  |  {folder}/{CHANNEL}\n"
        f"f0 = {f0:.5f} Hz  |  fs = {fs:.1f} Hz  |  "
        f"Attenuation = FFT peak reduction vs bandpassed at each harmonic",
        fontsize=11, fontweight='bold', y=0.98,
    )

    tbl = ax.table(cellText=rows_text, colLabels=col_labels,
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.6)

    for r_idx, (_, row_c) in enumerate(zip(rows_text, rows_color)):
        for c_idx, color in enumerate(row_c):
            tbl[r_idx + 1, c_idx].set_facecolor(color)
    for c_idx in range(n_cols):
        tbl[0, c_idx].set_facecolor('#1565C0')
        tbl[0, c_idx].set_text_props(color='white', fontweight='bold')
    tbl.auto_set_column_width(list(range(n_cols)))

    legend_items = [
        ('#A5D6A7', '>20 dB (excellent)'),
        ('#C8E6C9', '10-20 dB (good)'),
        ('#FFF9C4', '3-10 dB (marginal)'),
        ('#FFCDD2', '0-3 dB (poor)'),
        ('#EF9A9A', '<0 dB (amplification)'),
    ]
    lx = 0.01
    for color, label in legend_items:
        ax.add_patch(plt.Rectangle((lx, 0.01), 0.025, 0.035,
                                   transform=fig.transFigure, facecolor=color,
                                   edgecolor='grey', clip_on=False, linewidth=0.5))
        ax.text(lx + 0.030, 0.025, label,
                transform=fig.transFigure, fontsize=7.5, va='center')
        lx += 0.175

    if SAVE_FIGS:
        fig.savefig("filter_comparison_table.png", dpi=150, bbox_inches='tight')
        print("Saved -> filter_comparison_table.png")
    plt.show()


In [ ]:
def print_metrics(x_bp, filtered, timings, fs=44100.0, f0=F0):
    """Print console table: RMS, Atten@f0, Atten@2*f0, processing time."""
    def peak_at(x, f_target, f_bw=1.5):
        fr, mag = compute_fft(x, fs)
        mask = (fr >= f_target - f_bw) & (fr <= f_target + f_bw)
        return float(np.max(mag[mask])) if mask.any() else np.nan

    peak_bp_1 = peak_at(x_bp, f0)
    peak_bp_2 = peak_at(x_bp, 2 * f0)
    sep       = '-'

    print()
    print("  -- Harmonic attenuation & timing --")
    C1, C2, C3, C4, C5 = 'Filter', 'RMS', 'Atten@f0', 'Atten@2xf0', 'Time'
    print(f"  {C1:<20}  {C2:>10}  {C3:>12}  {C4:>12}  {C5:>8}")
    print(f"  {sep*20}  {sep*10}  {sep*12}  {sep*12}  {sep*8}")

    rms_bp = np.sqrt(np.mean(x_bp ** 2))
    ref    = '(ref)'
    print(f"  {'Bandpassed':<20}  {rms_bp:>10.6f}  {ref:>12}  {ref:>12}  {'---':>8}")

    for name, y in filtered.items():
        rms    = np.sqrt(np.mean(y ** 2))
        p1     = peak_at(y, f0)
        p2     = peak_at(y, 2 * f0)
        atten1 = 20 * np.log10(peak_bp_1 / (p1 + 1e-20))
        atten2 = 20 * np.log10(peak_bp_2 / (p2 + 1e-20))
        t_sec  = timings.get(name, float('nan'))
        print(f"  {name:<20}  {rms:>10.6f}  "
              f"{atten1:>+11.1f} dB  {atten2:>+11.1f} dB  {t_sec:>7.2f}s")

    print()
    print(f"  Atten@f0   : peak reduction at {f0:.5f} Hz vs bandpassed.")
    print(f"  Atten@2xf0 : peak reduction at {2*f0:.5f} Hz.")
    print("  RMS        : total signal energy -- should stay close to bandpassed.")
    print()
    print("  INTERPRETATION")
    print("  FIR Comb  RMS >> bandpassed: non-flat passband boosts mid-band EMG")
    print("    by up to +12 dB with filtfilt. Do NOT use as-is for clean EMG.")
    print("  IIR Comb  RMS ~= bandpassed: narrow 1.6 Hz notch removes only the")
    print("    powerline spike; EMG energy at harmonics is correctly preserved.")
    print("  IIR Notch RMS ~= bandpassed: best passband fidelity (Q=30, ~1.7 Hz).")
    print("  FIR Notch RMS < bandpassed : wide notch (~88 Hz) removes genuine EMG.")


def main(folder=DEFAULT_FOLDER):
    """Full pipeline: load -> bandpass -> 4 filters -> FFTs -> 6 figures + metrics."""
    t, x_raw, fs = load_recording(folder)
    f0 = F0

    print("Bandpass filtering 6-500 Hz ...")
    x_bp = bandpass_filter(x_raw, fs)

    filter_fns = {
        'FIR Notch': lambda x: fir_notch_filter(x, fs, f0=f0),
        'IIR Notch': lambda x: iir_notch_filter(x, fs, f0=f0),
        'FIR Comb':  lambda x: fir_comb_filter(x,  fs, f0=f0),
        'IIR Comb':  lambda x: iir_comb_filter(x,  fs, f0=f0),
    }
    filtered = {}
    timings  = {}
    for name, fn in filter_fns.items():
        print(f"Applying {name} ...")
        t0 = time.perf_counter()
        filtered[name] = fn(x_bp)
        timings[name]  = time.perf_counter() - t0
        print(f"  done in {timings[name]:.2f} s")

    print("Computing FFTs ...")
    ffts = {'raw': compute_fft(x_raw, fs), 'bandpassed': compute_fft(x_bp, fs)}
    for name, y in filtered.items():
        ffts[name] = compute_fft(y, fs)

    print("Computing single-stage frequency responses ...")
    responses = single_stage_responses(fs, f0=f0)

    print("Fig 1 -- main comparison ...")
    plot_demo(t, x_raw, x_bp, filtered, ffts, responses,
              fs=fs, f0=f0, folder=folder)

    print("Fig 2 -- full-signal overlay ...")
    plot_overlay(t, x_raw, x_bp, filtered, ffts, fs=fs, f0=f0, folder=folder)

    print("Fig 3 -- time-domain subplots ...")
    plot_subplots_time(t, x_raw, x_bp, filtered, fs=fs, f0=f0, folder=folder)

    print("Fig 4 -- FFT log-scale subplots ...")
    plot_subplots_fft(ffts, fs=fs, f0=f0, folder=folder)

    print("Fig 5 -- FFT linear-scale subplots (0-800 Hz) ...")
    plot_subplots_fft_linear(ffts, fs=fs, f0=f0, folder=folder)

    print("Fig 6 -- performance table ...")
    plot_performance_table(x_bp, filtered, timings, ffts,
                           fs=fs, f0=f0, folder=folder)

    print_metrics(x_bp, filtered, timings, fs=fs, f0=f0)


In [ ]:
# Scan for recording directories that contain the channel CSV
available_folders = sorted([
    d.name for d in RECORDINGS_DIR.iterdir()
    if d.is_dir() and (d / f"{CHANNEL}.csv").exists()
]) if RECORDINGS_DIR.exists() else []

if not available_folders:
    print(f"No recordings found in '{RECORDINGS_DIR}/' with {CHANNEL}.csv")
else:
    folder_dropdown = widgets.Dropdown(
        options=available_folders,
        value=DEFAULT_FOLDER if DEFAULT_FOLDER in available_folders else available_folders[0],
        description="Recording folder:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="450px"),
    )

    run_button = widgets.Button(
        description=" Run Analysis",
        button_style="primary",
        icon="play",
        tooltip="Load, filter, and produce all 6 figures + metrics table",
        layout=widgets.Layout(width="160px", height="36px"),
    )

    status_label = widgets.Label(
        value="Ready -- select a folder and click Run Analysis.",
        layout=widgets.Layout(margin="0 0 0 12px"),
    )

    out = widgets.Output()

    def on_run(b):
        run_button.disabled = True
        status_label.value = "Running... (FIR Notch takes ~10 s at fs=44100 Hz)"
        with out:
            clear_output(wait=True)
            try:
                main(folder_dropdown.value)
                status_label.value = f"Done -- {folder_dropdown.value}"
            except Exception as exc:
                import traceback
                traceback.print_exc()
                status_label.value = f"Error: {exc}"
        run_button.disabled = False

    run_button.on_click(on_run)

    display(widgets.VBox([
        widgets.HTML("<h3 style='margin:6px 0'>EMG Filter Comparison</h3>"),
        widgets.HBox([folder_dropdown, run_button, status_label]),
        out,
    ]))
